# Databricks `src` Import Smoke Test

Run this notebook in Databricks before refactoring the production training and inference notebooks. It verifies that Databricks can import the local `src/solar_yield` package and execute the shared feature helpers.

In [ ]:
import importlib
import importlib.machinery
import importlib.util
import subprocess
import sys
import types
from pathlib import Path


def find_repo_root():
    candidates = []

    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
        workspace_dir = workspace_path.parent
        candidates.extend([workspace_dir, *workspace_dir.parents])
    except Exception:
        pass

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "solar_yield" / "features.py").exists():
            return candidate

    searched = "\n".join(str(candidate) for candidate in candidates)
    raise RuntimeError(
        "Could not locate repo root containing src/solar_yield/features.py. "
        f"Searched:\n{searched}"
    )


def try_import_features():
    importlib.invalidate_caches()
    return importlib.import_module("solar_yield.features")


def install_manual_package_path(repo_root):
    package_dir = repo_root / "src" / "solar_yield"
    init_file = package_dir / "__init__.py"
    if not init_file.exists():
        raise FileNotFoundError(f"Missing expected package file: {init_file}")

    sys.modules.pop("solar_yield", None)
    package = types.ModuleType("solar_yield")
    package.__file__ = str(init_file)
    package.__path__ = [str(package_dir)]
    package.__package__ = "solar_yield"
    package.__spec__ = importlib.machinery.ModuleSpec(
        "solar_yield",
        loader=None,
        is_package=True,
    )
    package.__spec__.submodule_search_locations = [str(package_dir)]
    sys.modules["solar_yield"] = package
    try_import_features()


def ensure_solar_yield_importable(repo_root):
    src_path = str(repo_root / "src")
    repo_path = str(repo_root)
    package_dir = repo_root / "src" / "solar_yield"

    print(f"Package dir exists: {package_dir.exists()} -> {package_dir}")
    print(f"features.py exists: {(package_dir / 'features.py').exists()}")

    for path in (src_path, repo_path):
        if path not in sys.path:
            sys.path.insert(0, path)

    try:
        try_import_features()
        return "sys.path"
    except ModuleNotFoundError as error:
        print(f"Direct sys.path import failed: {error}")

    print("Trying local editable install with --no-deps from the repo root...")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(repo_root)]
        )
        try_import_features()
        return "editable install"
    except Exception as error:
        print(f"Editable install import failed: {type(error).__name__}: {error}")

    print("Trying manual package path fallback for Databricks workspace files...")
    install_manual_package_path(repo_root)
    return "manual package path"


repo_root = find_repo_root()
src_path = str(repo_root / "src")
import_strategy = ensure_solar_yield_importable(repo_root)

print(f"Repo root: {repo_root}")
print(f"Added to sys.path: {src_path}")
print(f"Import strategy: {import_strategy}")

In [ ]:
import numpy as np
import pandas as pd

from solar_yield.config import SiteConfig
from solar_yield.features import (
    FEATURE_COLUMNS,
    add_fraction_features,
    apply_physical_overrides,
    prepare_model_matrix,
)
from solar_yield.physics import project_to_plane_of_array


raw = pd.DataFrame(
    {
        "timestamp": pd.date_range("2026-01-01 08:00", periods=3, freq="h"),
        "sunshine_duration": [0.0, 1800.0, 3600.0],
        "cloud_low": [0.0, 50.0, 100.0],
        "cloud_mid": [10.0, 20.0, 30.0],
        "cloud_high": [20.0, 30.0, 40.0],
        "water_vapour": [11.0, 12.0, 13.0],
        "temperature": [21.0, 22.0, 23.0],
        "relative_humidity": [60.0, 55.0, 50.0],
        "surface_pressure": [1010.0, 1011.0, 1012.0],
    }
)

features = add_fraction_features(raw)
matrix = prepare_model_matrix(features)

assert list(matrix.columns) == FEATURE_COLUMNS
assert not matrix.isna().any().any()

predictions = np.array(
    [
        [0.6, 0.2],
        [0.8, 0.3],
        [1.2, -0.1],
    ]
)
scored = apply_physical_overrides(features, predictions)
assert {"final_pred_direct", "final_pred_diffuse"}.issubset(scored.columns)
assert scored.loc[0, "final_pred_direct"] == 0.0
assert scored.loc[2, "final_pred_direct"] == 0.0

SiteConfig().validate()
assert callable(project_to_plane_of_array)

print("SUCCESS: Databricks imported src/solar_yield and executed shared feature helpers.")
print(f"Feature columns ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")